In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(1)

In [22]:
# LSTM : 시간 순서가 있는 데이터를 처리하는 레이어
# 일반 신경망은 이전 입력을 기억 못하는데, LSTM은 기억

# 레이어 설계
# 단어는 보통 6차원 벡터로 표시되기 때문에 EMBEDDING_DIM을 6으로 할당하고
# 공정 같은 경우는 시점 마다 전류, RPM이 표시하는 숫자값이 배열 안에 각각의 형태로 들어올 것이기 때문에 특징 수가 2개이다.
lstm = nn.LSTM(3, 6)
#              ↑  ↑
#          특징수  기억크기

# 예시 데이터
inputs = [torch.randn(1, 1, 3) for _ in range(5)]
#                     ↑  ↑  ↑
#                   seq 배치 특징수
# 예를 들면
# 3번째 매개변수는 특징수이기 때문에 [전류, RPM, 온도]  이렇게 3개가 예시가 될 수 있다.

# torch.randn(50, 32, 2)
#             ↑   ↑   ↑
#          50시점 32개 특징수
#                 묶음
# 32개를 동시에 50시점씩 처리하는 것. 데이터 32개를 한 번에 넣는 것

hidden = (torch.rand(1, 1, 6),  # h_n : 단기 기억
          torch.rand(1, 1, 6))  # c_n : 장기 기억
#                    ↑  ↑  ↑
#   (형태 맞추려고) seq 배치/기억크기
# seq은 보통 1로 초기화
# batch 크기는 동일하게 맞추기

for i in inputs:
  # 레이어 실행
  out, hidden = lstm(i.view(1, 1, -1), hidden)
  #                  ↑                 ↑
  #                3D 입력형식          이전 기억
  # 3D로 만드는 이유는 lstm이 받을 수 있는 형태이기 때문이다.

# 위의 과정과 같지만 코드 줄을 줄이는 방법 (for문을 사용하지 않음)
# view는 시점들을 3차원으로 lstm이 받을 수 있는 형태로 변경시키는 것이다.
# 첫 번째 매개변수 : 시점의 총 개수를 전달
# 두 번째 매개변수 : batch를 확인하고 그대로 전달
# 세 번째 매개변수 : 보통 귀찮아서 -1을 쓰긴 하지만 제대로 넣고 싶으면 특징 수를 집어넣으면 된다.
inputs = torch.cat(inputs).view(len(inputs), 1, -1)
hidden = (torch.rand(1, 1, 6), torch.rand(1, 1, 6))

out, hidden = lstm(inputs, hidden)

In [23]:
# 단어를 숫자로 바꾸는 함수
# sequence : 순서 있는 데이터(LSTM의 핵심)
def prepare_sequence(seq, to_ix):
  idxs = [to_ix[w] for w in seq]
  return torch.tensor(idxs, dtype=torch.long)

training_data = [
  # 첫 번째 : 순서 있는 입력 데이터
  # 두 번째 : 각 입력에 대한 정답 레이블
  ("The dog ate the apple".split(), ["DET", "NN", "V", "DET", "NN"]),
  ("Everybody read that book".split(), ["NN", "V", "DET", "NN"])
]

# 단어를 index화
word_to_ix = {}

# 중복되는 단어를 제외하고 단어에 고유 번호를 부여
# 단어를 집어넣어 품사를 예측하기 위한 사전 작업용 반복문
for sent, tags in training_data:
  for word in sent:
    if word not in word_to_ix:
      word_to_ix[word] = len(word_to_ix)


print(word_to_ix)

# 품사를 숫자로 바꾸는 딕셔너리
tag_to_ix = {"DET": 0, "NN": 1, "V": 2}

# 단어를 몇 차원 벡터로 표현할지 크기
EMBEDDING_DIM = 6
# 기억 크기
HIDDEN_DIM = 6

{'The': 0, 'dog': 1, 'ate': 2, 'the': 3, 'apple': 4, 'Everybody': 5, 'read': 6, 'that': 7, 'book': 8}


In [ ]:
class LSTMTagger(nn.Module):
  def __init__(self, embedding_dim, hidden_dim, vocab_size, target_size):
    super().__init__()
    self.hidden_dim = hidden_dim

    # Embedding : 단어를 벡터로 바꾸는 것
    # 처음 초기화할 때는 의미 없는 랜덤값이지만 학습 후에는 의미 있는 벡터로 변한다.
    self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)

    # embedding_dim
    # 레이어 설계
    # batch를 첫 번째 축에 배치함으로써 혼란을 줄인다.
    self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)

    # 기억 크기를 품사의 정답에 맞게 압축
    # 기억 크기를 target_size(정답)의 크기 만큼 줄이기
    self.hidden2tag = nn.Linear(hidden_dim, target_size)

  def forward(self, sentence):
    # 단어는 숫자로 표현해야 하는 과정을 거쳐야 하기 때문에 embedding을 해준다.
    # sentence는 [0, 1, 2, 3, 4] 이런 형식
    embeds = self.word_embeddings(sentence)
    # embeds의 shape는 (len(sentence), 6) - word_embeddings로 각 시점들을 벡터화 완료
    # 보통 view에서 batch를 추가해준다.
    # hidden을 받을지 안 받을지는 데이터가 독립적인지 아닌지에 따라 달라진다.
    # 한 문장당 단어의 품사를 예측할 때는 각 문장이 독립적이어야 하니까. hidden이 필요없다.
    # 공정 예지보전은 센서 데이터가 연속적이기 때문에 hidden으로 기억을 유지시켜야 한다.
    # 순서 없음 → LSTM 쓸 필요 없음 (FashionMNIST)
    # 순서 있음 → LSTM 사용
    #   └ 독립적 → hidden 버림 (문장)
    #   └ 연속적 → hidden 유지 (센서)
    # 연속적 (센서) → 윈도우로 잘라서 + hidden 유지
    # 독립적 (문장) → 전체 시점 한 번에 + hidden 버림
    # batch_first 때문에 batch가 0번째 축으로 이동
    lstm_out, _ = self.lstm(embeds.view(1, len(sentence), -1))
    
    tag_space = self.hidden2tag(lstm_out)
    
    return tag_space

In [25]:
model = LSTMTagger(EMBEDDING_DIM, HIDDEN_DIM, len(word_to_ix), len(tag_to_ix))
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1)

for epoch in range(300):
  # 문장들을 꺼낸다.
  for sentence, tags in training_data:

    # 미리 준비된 단어별 고유한 인덱스와 준비된 문장을 넣어
    # 준비된 문장을 각 고유한 번호와 매치하여 새로운 배열 생성
    sentence_in = prepare_sequence(sentence, word_to_ix)
    # 마찬가지로 정답도 사전 준비한 정답 순서대로 정답 순서 인덱스 배열을 준비해놓는다.
    # 이 데이터는 sentence_in의 모델 통과값(예측값)과 비교하게 될 것이다.
    targets = prepare_sequence(tags, tag_to_ix)

    # 각 고유한 인덱스를 일단 6차원 벡터화하여 lstm이 읽을 수 있게 만든 뒤
    # view로 3차원 변환 후 lstm에서 기억을 형성한 뒤에
    # 예측값을 받는다.
    # lstm을 거치는 이유는 단어 순서로 품사를 예측할 것이기 때문이다.
    tag_space = model(sentence_in)

    # loss로 예측값과 정답을 비교한다.
    loss = loss_function(tag_space.view(-1, len(tag_to_ix)), targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# MNIST는 각 이미지가 독립적이지만, LSTM은 이전 것을 기억하면서 예측하는 게 핵심 차이
with torch.no_grad():
  inputs = prepare_sequence(training_data[0][0], word_to_ix)
  tag_space = model(inputs)

  print(tag_space)

tensor([[[ 1.7598, -0.4955, -1.4255],
         [-1.7286,  2.6318, -0.9117],
         [-0.4477, -1.2372,  2.0508],
         [ 2.5924, -1.0318, -0.9384],
         [-1.0168,  2.7874, -1.5324]]])
